In [1]:
import os
import sys
import re
import numpy as np
import pandas as pd
import plotly.express as px
import scipy.stats as stats
from scipy.stats import pearsonr
from dash import html, dcc, Input, Output, Dash
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sys.path.append(os.path.join(os.getcwd(), '..', '..', '..'))
from baseVR.base_functionality import init_import_paths
init_import_paths() 

from CustomLogger import CustomLogger as Logger
from analytics_processing import analytics

from analytics_processing.sessions_from_nas_parsing import sessionlist_fullfnames_from_args, fullfnames2snames
from dashsrc.plot_components.plots import plot_TrackFiringRate
from dashsrc.plot_components.plots import plot_unit_fr_stability


In [2]:
Logger().init_logger(None, None, logging_level="DEBUG")
animal_ids = [6]
paradigm = [1100]
session_range = [1,33]
session_ids = None
normalize = True
smooth = False
excl_session_names =  ['2024-11-29_17-21_rYL006_P1100_LinearTrackStop_28min', '2025-01-21_18-49_rYL006_P1100_LinearTrackStop_30min', '2025-01-22_17-51_rYL006_P1100_LinearTrackStop_5min', '2024-12-11_17-42_rYL006_P1100_LinearTrackStop_30min'] # ignore short sessions (10, 24,25)

session_dirs = sessionlist_fullfnames_from_args(paradigm, animal_ids, session_ids, excl_session_names=excl_session_names)[0]
session_names= fullfnames2snames(session_dirs)

2026-02-26 16:41:51,337|DEBUG|41770|sessions_from_nas_parsing|get_sessionlist_fullfnames
	Searching NAS for applicable sessions...
2026-02-26 16:41:51,693|DEBUG|41770|sessions_from_nas_parsing|get_sessionlist_fullfnames
	Session 2024-11-29_17-21_rYL006_P1100_LinearTrackStop_28min.hdf5 excluded
2026-02-26 16:41:51,974|DEBUG|41770|sessions_from_nas_parsing|get_sessionlist_fullfnames
	Session 2024-12-11_17-42_rYL006_P1100_LinearTrackStop_30min.hdf5 excluded
2026-02-26 16:41:52,151|DEBUG|41770|sessions_from_nas_parsing|get_sessionlist_fullfnames
	Session 2025-01-21_18-49_rYL006_P1100_LinearTrackStop_30min.hdf5 excluded
2026-02-26 16:41:52,174|DEBUG|41770|sessions_from_nas_parsing|get_sessionlist_fullfnames
	Session 2025-01-22_17-51_rYL006_P1100_LinearTrackStop_5min.hdf5 excluded
2026-02-26 16:41:52,418|DEBUG|41770|sessions_from_nas_parsing|get_sessionlist_fullfnames
	For paradigms [1100], animals [6], found 30 sessions.
2026-02-26 16:41:52,419|DEBUG|41770|sessions_from_nas_parsing|sessionl

In [3]:
# firing rates and behavior data
fr = analytics.get_analytics('FiringRate40msHz', session_names=session_names)
fr_track = analytics.get_analytics('FiringRateTrackwiseHz', session_names=session_names)
# fr_z_scored  = analytics.get_analytics('FiringRate40msZ', session_names=session_names)
fr_z_all_sess = fr.apply(lambda unit_fr: ((unit_fr - unit_fr.mean()) / unit_fr.std()))
behav = analytics.get_analytics('BehaviorTrackwise', session_names=session_names)
# behav.index = behav.index.droplevel(('animal_id', 'paradigm_id', 'entry_id'))

2026-02-26 16:41:52,431|DEBUG|41770|sessions_from_nas_parsing|sessionnames2fullfnames
	Inferring NAS paths for list of session names...
2026-02-26 16:41:52,885|DEBUG|41770|sessions_from_nas_parsing|sessionlist_fullfnames_from_args
	Paradigm_ids: None, animal_ids: None, session_ids: None, from_date: None, to_date: None
	Merging 30 sessions

2026-02-26 16:41:52,886|DEBUG|41770|analytics|get_analytics
	Processing FiringRate40msHz, (1100, 6, '2024-11-14_15-01') 2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min.hdf5
/Volumes/large/BMI/VirtualReality/SpatialSequenceLearning/RUN_rYL006/rYL006_P1100/2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min
2026-02-26 16:41:52,904|INFO|41770|analytics|get_analytics
	Analytic `FiringRate40msHz` does not exist for (1100, 6, '2024-11-14_15-01'), compute first, or check for typo
2026-02-26 16:41:52,905|DEBUG|41770|analytics|get_analytics
	Processing FiringRate40msHz, (1100, 6, '2024-11-14_16-40') 2024-11-14_16-40_rYL006_P1100_LinearTrackStop_21min.hdf5

In [4]:
behav.keys()

Index(['from_position_bin', 'to_position_bin', 'posbin_position',
       'posbin_velocity', 'posbin_acc', 'posbin_raw', 'posbin_yaw',
       'posbin_pitch', 'posbin_raw_500msMedian', 'posbin_yaw_500msMedian',
       'posbin_pitch_500msMedian', 'posbin_raw_abs_acc_500msMedian',
       'posbin_yaw_abs_acc_500msMedian', 'posbin_pitch_abs_acc_500msMedian',
       'posbin_RawYawPitch_abs_vel_sum',
       'posbin_RawYawPitch_abs_vel_sum_500msMedian',
       'posbin_RawYawPitch_abs_acc_sum_500msMedian',
       'posbin_YawPitch_abs_vel_sum_500msMedian',
       'posbin_YawPitch_abs_acc_sum_500msMedian', 'forward_vs_rotation_corr',
       'posbin_forward_prop', 'posbin_below_velocity_thr',
       'facecam_pose_nose_neck_body1_angle_velocity',
       'facecam_pose_body1_body2_body3_angle_velocity',
       'velocity_threshold_at_R1', 'velocity_threshold_at_R2',
       'facecam_pose_nose_x', 'facecam_pose_nose_y',
       'facecam_pose_nose_likelihood', 'facecam_pose_neck_x',
       'facecam_pose_ne

In [5]:
t0_events = analytics.get_analytics('TrialWiseT0Events40ms', session_names=session_names, mode = 'recompute' )
t0_events

2026-02-26 16:43:26,190|DEBUG|41770|sessions_from_nas_parsing|sessionnames2fullfnames
	Inferring NAS paths for list of session names...
2026-02-26 16:43:26,603|DEBUG|41770|sessions_from_nas_parsing|sessionlist_fullfnames_from_args
	Paradigm_ids: None, animal_ids: None, session_ids: None, from_date: None, to_date: None
	Merging 30 sessions

2026-02-26 16:43:26,604|DEBUG|41770|analytics|get_analytics
	Processing TrialWiseT0Events40ms, (1100, 6, '2024-11-14_15-01') 2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min.hdf5
/Volumes/large/BMI/VirtualReality/SpatialSequenceLearning/RUN_rYL006/rYL006_P1100/2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min
2026-02-26 16:43:26,620|DEBUG|41770|analytics|_compute_sess_analytic
	Computing TrialWiseT0Events40ms for 2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min:
2026-02-26 16:43:26,621|DEBUG|41770|sessions_from_nas_parsing|sessionnames2fullfnames
	Inferring NAS paths for list of session names...
2026-02-26 16:43:26,623|DEBUG|41770|sessions_fr

In [6]:
behav['reward-valve-open_detected']

paradigm_id  animal_id  session_id        entry_id
1100         6          2024-11-14_15-01  0           0.0
                                          1           0.0
                                          2           0.0
                                          3           0.0
                                          4           0.0
                                                     ... 
                        2025-01-27_13-39  65051       0.0
                                          65052       0.0
                                          65053       0.0
                                          65054       0.0
                                          65055       0.0
Name: reward-valve-open_detected, Length: 1442177, dtype: float64

In [7]:
behavior = analytics.get_analytics(analytic="Behavior40msAligned", session_names=session_names)
behavior

2026-02-26 16:44:08,640|DEBUG|41770|sessions_from_nas_parsing|sessionnames2fullfnames
	Inferring NAS paths for list of session names...
2026-02-26 16:44:08,904|DEBUG|41770|sessions_from_nas_parsing|sessionlist_fullfnames_from_args
	Paradigm_ids: None, animal_ids: None, session_ids: None, from_date: None, to_date: None
	Merging 30 sessions

2026-02-26 16:44:08,906|DEBUG|41770|analytics|get_analytics
	Processing Behavior40msAligned, (1100, 6, '2024-11-14_15-01') 2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min.hdf5
/Volumes/large/BMI/VirtualReality/SpatialSequenceLearning/RUN_rYL006/rYL006_P1100/2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min
2026-02-26 16:44:08,919|INFO|41770|analytics|get_analytics
	Analytic `Behavior40msAligned` does not exist for (1100, 6, '2024-11-14_15-01'), compute first, or check for typo
2026-02-26 16:44:08,922|DEBUG|41770|analytics|get_analytics
	Processing Behavior40msAligned, (1100, 6, '2024-11-14_16-40') 2024-11-14_16-40_rYL006_P1100_LinearTrackStop_2

frame_raw_500msMedian  \
paradigm_id animal_id session_id       entry_id                          
1100        6         2024-11-14_16-40 0                           NaN   
                                       1                           NaN   
                                       2                           NaN   
                                       3                           NaN   
                                       4                           NaN   
...                                                                ...   
                      2025-01-27_13-39 110898                      NaN   
                                       110899                      NaN   
                                       110900                      NaN   
                                       110901                      NaN   
                                       110902                      NaN   

                                                 frame_raw_abs_acc_500msMedian  \
paradigm_id animal_id session_id       entry_id                                  
1100        6         2024-11-14_16-40 0                                   NaN   
                                       1                                   NaN   
                                       2                                   NaN   
                                       3                                   NaN   
                                       4                                   NaN   
...                                                                        ...   
                      2025-01-27_13-39 110898                              NaN   
                                       110899                              NaN   
                                       110900                              NaN   
                                       110901                              NaN   
                                       110902                              NaN   

                                                 frame_YawPitch_abs_vel_sum_500msMedian  \
paradigm_id animal_id session_id       entry_id                                           
1100        6         2024-11-14_16-40 0                                            NaN   
                                       1                                            NaN   
                                       2                                            NaN   
                                       3                                            NaN   
                                       4                                            NaN   
...                                                                                 ...   
                      2025-01-27_13-39 110898                                       NaN   
                                       110899                                       NaN   
                                       110900                                       NaN   
                                       110901                                       NaN   
                                       110902                                       NaN   

                                                 frame_YawPitch_abs_acc_sum_500msMedian  \
paradigm_id animal_id session_id       entry_id                                           
1100        6         2024-11-14_16-40 0                                            NaN   
                                       1                                            NaN   
                                       2                                            NaN   
                                       3                                            NaN   
                                       4                                            NaN   
...                                                                                 ...   
                      2025-01-27_13-39 110898                                       NaN   
                

In [8]:
behavior.keys()

Index(['frame_raw_500msMedian', 'frame_raw_abs_acc_500msMedian',
       'frame_YawPitch_abs_vel_sum_500msMedian',
       'frame_YawPitch_abs_acc_sum_500msMedian',
       'frame_RawYawPitch_abs_vel_sum_500msMedian',
       'frame_RawYawPitch_abs_acc_sum_500msMedian', 'frame_forward_prop',
       'forward_vs_rotation_corr', 'frame_position',
       'facecam_pose_nose_neck_body1_angle_velocity',
       'facecam_pose_nose_neck_body1_angle',
       'facecam_pose_nose_neck_body1_angle_likelihood',
       'frame_ephys_timestamp', 'frame_pc_timestamp', 'trial_id', 'cue',
       'trial_outcome', 'choice_R1', 'choice_R2', 'lick_detected',
       'reward-sound_detected', 'reward-valve-open_detected', 'track_zone',
       'from_ephys_timestamp', 'to_ephys_timestamp'],
      dtype='object')

In [9]:
reward_timings = behavior[behavior['reward-valve-open_detected'] == True]
reward_timings

frame_raw_500msMedian  \
paradigm_id animal_id session_id       entry_id                          
1100        6         2024-11-14_16-40 690                   14.732325   
                                       1347                  22.936904   
                                       6379                  12.859367   
                                       11081                  2.205580   
                                       11198                 19.452242   
...                                                                ...   
                      2025-01-27_13-39 99281                  4.348683   
                                       100297                 0.832736   
                                       102590                 0.339174   
                                       106862                14.668530   
                                       109991                 5.217457   

                                                 frame_raw_abs_acc_500msMedian  \
paradigm_id animal_id session_id       entry_id                                  
1100        6         2024-11-14_16-40 690                          310.168217   
                                       1347                           0.000000   
                                       6379                           7.795358   
                                       11081                         11.072584   
                                       11198                          7.710925   
...                                                                        ...   
                      2025-01-27_13-39 99281                         36.065566   
                                       100297                         6.237060   
                                       102590                        24.522804   
                                       106862                        31.236423   
                                       109991                        86.699173   

                                                 frame_YawPitch_abs_vel_sum_500msMedian  \
paradigm_id animal_id session_id       entry_id                                           
1100        6         2024-11-14_16-40 690                                     1.461005   
                                       1347                                   10.432527   
                                       6379                                   24.825595   
                                       11081                                   0.726808   
                                       11198                                   9.221894   
...                                                                                 ...   
                      2025-01-27_13-39 99281                                   5.339812   
                                       100297                                  4.755868   
                                       102590                                  6.674218   
                                       106862                                  5.348795   
                                       109991                                  7.163785   

                                                 frame_YawPitch_abs_acc_sum_500msMedian  \
paradigm_id animal_id session_id       entry_id                                           
1100        6         2024-11-14_16-40 690                                    36.401281   
                                       1347                                   24.777545   
                                       6379                                   45.513830   
                                       11081                                   4.830232   
                                       11198                                   6.281803   
...                                                                                 ...   
                      2025-01-27_13-39 99281                                  20.600760   
                

In [10]:
t0_ens = analytics.get_analytics('EnsembleT0Projection', session_names=session_names, mode='recompute')

2026-02-26 16:44:32,807|DEBUG|41770|sessions_from_nas_parsing|sessionnames2fullfnames
	Inferring NAS paths for list of session names...
2026-02-26 16:44:32,867|DEBUG|41770|sessions_from_nas_parsing|sessionlist_fullfnames_from_args
	Paradigm_ids: None, animal_ids: None, session_ids: None, from_date: None, to_date: None
	Merging 30 sessions

2026-02-26 16:44:32,868|DEBUG|41770|analytics|get_analytics
	Processing EnsembleT0Projection, (1100, 6, '2024-11-14_15-01') 2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min.hdf5
/Volumes/large/BMI/VirtualReality/SpatialSequenceLearning/RUN_rYL006/rYL006_P1100/2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min
2026-02-26 16:44:32,873|DEBUG|41770|analytics|_compute_sess_analytic
	Computing EnsembleT0Projection for 2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min:
2026-02-26 16:44:32,873|DEBUG|41770|sessions_from_nas_parsing|sessionnames2fullfnames
	Inferring NAS paths for list of session names...
2026-02-26 16:44:32,874|DEBUG|41770|sessions_from

/Volumes/large/BMI/VirtualReality/SpatialSequenceLearning/RUN_rYL006/rYL006_P1100/2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min/../../animal_analytics/ConcatenatedEnsambleProj40ms.parquet


2026-02-26 16:45:11,402|INFO|41770|analytics|get_analytics
	Loaded animal-level analytic `ConcatenatedEnsambleProj40ms` last modified on 2026-02-05T15:20:02.204022
2026-02-26 16:45:11,405|DEBUG|41770|sessions_from_nas_parsing|sessionnames2fullfnames
	Inferring NAS paths for list of session names...
2026-02-26 16:45:11,425|DEBUG|41770|sessions_from_nas_parsing|sessionlist_fullfnames_from_args
	Paradigm_ids: None, animal_ids: None, session_ids: None, from_date: None, to_date: None
	Merging 1 sessions

2026-02-26 16:45:11,425|DEBUG|41770|analytics|get_analytics
	Processing TrialWiseT0Events40ms, (1100, 6, '2024-11-14_15-01') 2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min.hdf5
/Volumes/large/BMI/VirtualReality/SpatialSequenceLearning/RUN_rYL006/rYL006_P1100/2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min
2026-02-26 16:45:11,440|INFO|41770|analytics|get_analytics
	Analytic `TrialWiseT0Events40ms` does not exist for (1100, 6, '2024-11-14_15-01'), compute first, or check for typo
202

/Volumes/large/BMI/VirtualReality/SpatialSequenceLearning/RUN_rYL006/rYL006_P1100/2024-11-14_16-40_rYL006_P1100_LinearTrackStop_21min/../../animal_analytics/ConcatenatedEnsambleProj40ms.parquet


2026-02-26 16:45:14,946|INFO|41770|analytics|get_analytics
	Loaded animal-level analytic `ConcatenatedEnsambleProj40ms` last modified on 2026-02-05T15:20:02.204022
2026-02-26 16:45:14,946|DEBUG|41770|sessions_from_nas_parsing|sessionnames2fullfnames
	Inferring NAS paths for list of session names...
2026-02-26 16:45:14,953|DEBUG|41770|sessions_from_nas_parsing|sessionlist_fullfnames_from_args
	Paradigm_ids: None, animal_ids: None, session_ids: None, from_date: None, to_date: None
	Merging 1 sessions

2026-02-26 16:45:14,953|DEBUG|41770|analytics|get_analytics
	Processing TrialWiseT0Events40ms, (1100, 6, '2024-11-14_16-40') 2024-11-14_16-40_rYL006_P1100_LinearTrackStop_21min.hdf5
/Volumes/large/BMI/VirtualReality/SpatialSequenceLearning/RUN_rYL006/rYL006_P1100/2024-11-14_16-40_rYL006_P1100_LinearTrackStop_21min

2026-02-26 16:45:14,991|INFO|41770|analytics|get_analytics
	Returning TrialWiseT0Events40ms for 1 sessions.
2026-02-26 16:45:15,002|DEBUG|41770|analytics|get_analytics
	Paradigm_

...

2026-02-26 16:46:46,000|DEBUG|41770|analytics|_compute_sess_analytic
	Computed analytic EnsembleT0Projection:
                                                 from_ephys_timestamp  \
session_id       paradigm_id animal_id entry_id                         
2025-01-26_21-48 1100        6         0                      7960000   
                                       0                      8000000   
                                       0                      8040000   
                                       0                      8080000   
                                       0                      8120000   
...                                                               ...   
                                       1246                3349160000   
                                       1246                3349200000   
                                       1246                3349240000   
                                       1246                3349280000   
              

/Volumes/large/BMI/VirtualReality/SpatialSequenceLearning/RUN_rYL006/rYL006_P1100/2025-01-27_13-39_rYL006_P1100_LinearTrackStop_73min/../../animal_analytics/ConcatenatedEnsambleProj40ms.parquet


2026-02-26 16:47:11,452|INFO|41770|analytics|get_analytics
	Loaded animal-level analytic `ConcatenatedEnsambleProj40ms` last modified on 2026-02-05T15:20:02.204022
2026-02-26 16:47:11,455|DEBUG|41770|sessions_from_nas_parsing|sessionnames2fullfnames
	Inferring NAS paths for list of session names...
2026-02-26 16:47:11,464|DEBUG|41770|sessions_from_nas_parsing|sessionlist_fullfnames_from_args
	Paradigm_ids: None, animal_ids: None, session_ids: None, from_date: None, to_date: None
	Merging 1 sessions

2026-02-26 16:47:11,465|DEBUG|41770|analytics|get_analytics
	Processing TrialWiseT0Events40ms, (1100, 6, '2025-01-27_13-39') 2025-01-27_13-39_rYL006_P1100_LinearTrackStop_73min.hdf5
/Volumes/large/BMI/VirtualReality/SpatialSequenceLearning/RUN_rYL006/rYL006_P1100/2025-01-27_13-39_rYL006_P1100_LinearTrackStop_73min

2026-02-26 16:47:11,514|INFO|41770|analytics|get_analytics
	Returning TrialWiseT0Events40ms for 1 sessions.
2026-02-26 16:47:11,523|DEBUG|41770|analytics|get_analytics
	Paradigm_

No data for session: 2024-11-14_16-40
No data for session: 2024-11-15_15-48
No data for session: 2024-11-20_17-46
No data for session: 2024-11-21_17-22
No data for session: 2024-11-25_16-25
No data for session: 2024-11-26_16-39
No data for session: 2024-11-28_17-41
No data for session: 2024-11-29_17-21
No data for session: 2024-12-02_16-09
No data for session: 2024-12-03_16-23
No data for session: 2024-12-04_18-06
No data for session: 2024-12-06_16-49
No data for session: 2024-12-09_17-45
No data for session: 2024-12-10_17-20
No data for session: 2024-12-11_17-42
No data for session: 2024-12-12_16-13
No data for session: 2024-12-13_17-10
No data for session: 2025-01-14_18-08
No data for session: 2025-01-15_17-18
No data for session: 2025-01-16_17-47
No data for session: 2025-01-17_16-55
No data for session: 2025-01-21_18-49
No data for session: 2025-01-22_17-51
No data for session: 2025-01-23_16-48
No data for session: 2025-01-24_12-24
No data for session: 2025-01-24_19-37
No data for 

2026-02-26 16:47:13,723|DEBUG|41770|analytics|_compute_sess_analytic
	Computed analytic EnsembleT0Projection:
                                                 from_ephys_timestamp  \
session_id       paradigm_id animal_id entry_id                         
2025-01-27_13-39 1100        6         0                      8000000   
                                       0                      8040000   
                                       0                      8080000   
                                       0                      8120000   
                                       0                      8160000   
...                                                               ...   
                                       1108                4402200000   
                                       1108                4402240000   
                                       1108                4402280000   
                                       1108                4402320000   
              

In [11]:
t0_ens = analytics.get_analytics('EnsembleT0Projection', session_names=session_names, )
t0_ens

2026-02-26 17:06:14,488|DEBUG|41770|sessions_from_nas_parsing|sessionnames2fullfnames
	Inferring NAS paths for list of session names...
2026-02-26 17:06:14,832|DEBUG|41770|sessions_from_nas_parsing|sessionlist_fullfnames_from_args
	Paradigm_ids: None, animal_ids: None, session_ids: None, from_date: None, to_date: None
	Merging 30 sessions

2026-02-26 17:06:14,833|DEBUG|41770|analytics|get_analytics
	Processing EnsembleT0Projection, (1100, 6, '2024-11-14_15-01') 2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min.hdf5
/Volumes/large/BMI/VirtualReality/SpatialSequenceLearning/RUN_rYL006/rYL006_P1100/2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min
2026-02-26 17:06:14,880|INFO|41770|analytics|get_analytics
	Analytic `EnsembleT0Projection` does not exist for (1100, 6, '2024-11-14_15-01'), compute first, or check for typo
2026-02-26 17:06:14,881|DEBUG|41770|analytics|get_analytics
	Processing EnsembleT0Projection, (1100, 6, '2024-11-14_16-40') 2024-11-14_16-40_rYL006_P1100_LinearTrackSto

from_ephys_timestamp  \
paradigm_id animal_id session_id       entry_id                         
1100        6         2024-11-14_16-40 0                      4400000   
                                       1                      4440000   
                                       2                      4480000   
                                       3                      4520000   
                                       4                      4560000   
...                                                               ...   
                      2025-01-27_13-39 37995               4402200000   
                                       37996               4402240000   
                                       37997               4402280000   
                                       37998               4402320000   
                                       37999               4402360000   

                                                 to_ephys_timestamp  \
paradigm_id animal_id session_id       entry_id                       
1100        6         2024-11-14_16-40 0                    4440000   
                                       1                    4480000   
                                       2                    4520000   
                                       3                    4560000   
                                       4                    4600000   
...                                                             ...   
                      2025-01-27_13-39 37995             4402240000   
                                       37996             4402280000   
                                       37997             4402320000   
                                       37998             4402360000   
                                       37999             4402400000   

                                                 Assembly001  Assembly002  \
paradigm_id animal_id session_id       entry_id                             
1100        6         2024-11-14_16-40 0            0.044591    -0.079732   
                                       1            0.353522    -0.208656   
                                       2           -0.090020    -0.381483   
                                       3           -0.082682    -0.003392   
                                       4            0.169519    -0.213003   
...                                                      ...          ...   
                      2025-01-27_13-39 37995        0.018032    -0.425217   
                                       37996       -0.161524     0.784179   
                                       37997       -0.053947    -0.126021   
                                       37998       -0.077455    -0.351810   
                                       37999        0.005057     0.036111   

                                                 Assembly003  Assembly004  \
paradigm_id animal_id session_id       entry_id                             
1100        6         2024-11-14_16-40 0            0.117157    -0.100619   
                                       1           -0.050525    -0.098752   
                                       2            0.185447     1.103287   
                                       3            0.093962    -0.152654   
                                       4           -0.009496    -0.107002   
...                                                      ...          ...   
                      2025-01-27_13-39 37995        0.453529    -0.392653   
                                       37996        0.100941    -0.860001   
                                       37997        0.913100     1.616778   
                                       37998        4.636653    -0.125254   
                                       37999        0.180242     0.439290   

                                                 Assembly005  Assembly006  \
paradigm_id animal_id session_id       entry_id                             
1100        6    